In [ ]:
import pandas as pd
import json
import os
import numpy as np
from datasets import Dataset

from transformers import (
    set_seed,
)

from time import time
import pickle
import matplotlib.pyplot as plt
import random
from tqdm import tqdm
from unsloth import FastLanguageModel
from sklearn.metrics import roc_curve, roc_auc_score, accuracy_score, precision_score, recall_score, f1_score
import torch

def_seed = 42

set_seed(def_seed)
np.random.seed(def_seed)
import random
random.seed(def_seed)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig

from sklearn.model_selection import train_test_split


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


Selecting which task: 2 labels or 3 labels task

Loading synthetic data for labeling: to preserve anonymity, the links to Hugging Face have been removed, but a portion of 10,000 samples from each dataset is included.

In [3]:
no_Labels = 2
no_Labels = 3

In [ ]:
dataset_2 = load_dataset("anonymousOWSHateLLM/Hate.2_labels_labeled")
df_2_label = pd.DataFrame(dataset_2["2_labels"])

dataset_3 = load_dataset("anonymousOWSHateLLM/Hate.3_labels_labeled")
df_3_label = pd.DataFrame(dataset_3["3_labels"])
# df_2_label = pd.read_csv('data/2_label.csv')
# df_3_label = pd.read_csv('data/3_label.csv')

if no_Labels == 2:
    df = df_2_label
    prompt_template = '''You are tasked with annotating speech. Your response must be a single valid number:
    1 for Hate/Offensive/Sexism/Toxic/Political/COVID-related Hate Speech,
    2 for Neutral Speech.

    Provide only the number corresponding to the category. Do not include any explanation or additional text.
    Do you think the following comment is Hate/Offensive/Sexism/Toxic/Political/COVID-related Hate Speech, speech or Neutral speech?
    \n"{comment}"\n
    Your Answer:
    '''
else:
    df = df_3_label
    prompt_template = '''You are tasked with annotating speech. Your response must be a single valid number:
        1 for Hate Speech.
        2 for Offensive Speech.
        3 for Neutral Speech.

        Think step by step:
        - Identify whether the comment contains explicit hate towards a group or individual based on identity characteristics (Hate Speech).
        - If the comment is aggressive, rude, or contains insults but lacks clear hateful intent, classify it as Offensive Speech.
        - If the comment is neutral or non-offensive, classify it as Neutral Speech.

        Examples:
        Comment: "People of [group] are a disease and should be removed."
        Your Answer: 1

        Comment: "You're so stupid, just shut up!"
        Your Answer: 2

        Comment: "I hope you have a great day!"
        Your Answer: 3

        Provide only the number corresponding to the category. Do not include any explanation or additional text or your thinking process.
        Do you think the following comment is hate speech or offensive speech?
        \n"{comment}"\n
        Your Answer:
        '''


README.md:   0%|          | 0.00/1.18k [00:00<?, ?B/s]

3_labels-00000-of-00001.parquet:   0%|          | 0.00/56.7M [00:00<?, ?B/s]

Generating 3_labels split:   0%|          | 0/350364 [00:00<?, ? examples/s]

In [6]:
df_3_label.Mean.value_counts()

Mean
3    307293
2     40747
1      2324
Name: count, dtype: int64

In [10]:
df_3_label.Lgb.value_counts()

Lgb
3    330908
2     19246
1       210
Name: count, dtype: int64

Select which data to finetune: Lgb, Mean, Vote or Human data

In [5]:
df_3_label.columns

Index(['text', 'unsloth/gemma-2-9b-it-bnb-4bit_label_1',
       'unsloth/gemma-2-9b-it-bnb-4bit_label_2',
       'unsloth/Qwen2.5-14B-Instruct-bnb-4bit_label_1',
       'unsloth/Qwen2.5-14B-Instruct-bnb-4bit_label_2',
       'unsloth/mistral-7b-instruct-v0.3-bnb-4bit_label_1',
       'unsloth/mistral-7b-instruct-v0.3-bnb-4bit_label_2', 'language',
       'unsloth/Qwen2.5-14B-Instruct-bnb-4bit_label_3',
       'unsloth/mistral-7b-instruct-v0.3-bnb-4bit_label_3',
       'unsloth/gemma-2-9b-it-bnb-4bit_label_3', 'Lgb', '_label_1_mean',
       '_label_2_mean', '_label_3_mean', 'Mean'],
      dtype='object')

In [ ]:
df['label_id'] = df['Lgb']
# df['label_id'] = df['Mean']
# if "Vote" in df.columns:
#     df['label_id'] = df['Vote']
df['label_id'].value_counts()

Human data for finetune 2 labels: 80% of 84k human label data, include 7 dataset

In [ ]:
if no_Labels == 2:
    dataset = load_dataset("anonymousOWSHateLLM/7_human_dataset")
    human_df  = pd.DataFrame(dataset['main'])
    human_df = pd.read_csv('7_human_dataset.csv')
    human_df['ds'].value_counts()

    train_df, test_df = train_test_split(human_df, test_size=0.2, random_state=42)
    print(f"Train size: {len(train_df)}, Test size: {len(test_df)}")

    df = train_df
    train_df.label_id.value_counts()

    
    # df = train_df


README.md:   0%|          | 0.00/420 [00:00<?, ?B/s]

main-00000-of-00001.parquet:   0%|          | 0.00/6.60M [00:00<?, ?B/s]

Generating main split:   0%|          | 0/85918 [00:00<?, ? examples/s]

Train size: 68734, Test size: 17184


label_id
2    48114
1    20620
Name: count, dtype: int64

Human data for 3 Labels, 95% HateXplain and HateOff data

In [ ]:
if no_Labels == 3:
    # dataset = load_dataset("")
    # Lgb_df_3_labels  = pd.DataFrame(dataset['lgb_data_3_label'])
    Lgb_df_3_labels = pd.read_csv('lgb_data_3_label.csv')
    Lgb_df_3_labels.label_id.value_counts()

    train_df, test_df = train_test_split(Lgb_df_3_labels
    , test_size=0.05, random_state=42)
    print(train_df.label_id.value_counts())
    #     df = train_df

label_id
2    23376
3    11338
1     7822
Name: count, dtype: int64


In [10]:
def process_task(texts, model, tokenizer, stop_token_id):
    encoding = tokenizer(texts, padding=True, return_tensors='pt').to('cuda')
    with torch.no_grad():
        outputs = model(**encoding)
        logits = outputs.logits  
    last_token_logits = logits[:, -1, :] 
    probabilities = torch.softmax(last_token_logits, dim=-1)
    indices = torch.tensor(stop_token_id)
    probs = []
    for i in indices:
        probs.append( probabilities[:, i].float().cpu().numpy())
    return probs

def preprocess(text, label, model_id, tokenizer):
    user_message_content = prompt_template.format(comment=text)
    user_message = {
        "role": "user",
        "content": user_message_content
    }

    assistant_message = {
        "role": "assistant",
        "content": str(label)
    }

    if "Qwen" in model_id:
        system_message =  {"role": "system", "content": "You are Qwen, created by Alibaba Cloud. You are a helpful assistant"}
    else:
        system_message =  {"role": "system", "content": "You are a helpful assistant"}
    if "gemma" in model_id:
        messages = [user_message, assistant_message]
        messages = tokenizer.apply_chat_template(messages, tokenize=False)

    else:
        messages = [system_message, user_message, assistant_message]
    messages = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    messages = messages
    
    return messages


In [11]:
model_id ="unsloth/Qwen2.5-14B-Instruct-bnb-4bit"
model_id = "unsloth/Llama-3.2-1B-Instruct-bnb-4bit"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_id,
    max_seq_length = 500,
    dtype = getattr(torch, "bfloat16"),
)

df["prompt"] = df.apply(lambda row: preprocess(row["text"], row["label_id"],  model_id, tokenizer), axis=1)


In [12]:
print(df.prompt[0])

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 30 Apr 2025

You are a helpful assistant<|eot_id|><|start_header_id|>user<|end_header_id|>

You are tasked with annotating speech. Your response must be a single valid number:
    1 for Hate/Offensive/Sexism/Toxic/Political/COVID-related Hate Speech,
    2 for Normal Speech.

    Provide only the number corresponding to the category. Do not include any explanation or additional text.
    Do you think the following comment is Hate/Offensive/Sexism/Toxic/Political/COVID-related Hate Speech, speech or Normal speech?
    
"Guten Tag, 

ist das wahr, dass man Minecraft irgendwie kostenlos haben kann?
Wenn ja, wie und wo?

LG"

    Your Answer:<|eot_id|><|start_header_id|>assistant<|end_header_id|>

2<|eot_id|><|start_header_id|>assistant<|end_header_id|>




# Finetune Lora

In [ ]:
# dataset = Dataset.from_pandas(df.sample(n=2000))
dataset = Dataset.from_pandas(df.sample(frac=1, random_state=def_seed))


dataset = dataset.shuffle(seed=42)


output_dir = "Lgb.Llama1B"
sft_config = SFTConfig(
    output_dir=output_dir, 
    eval_strategy="no", 
    do_eval=False,  
    optim="paged_adamw_8bit", 
    per_device_train_batch_size=8, 
    gradient_accumulation_steps=8,  
    per_device_eval_batch_size=1,  
    log_level="debug",  
    save_steps=1000,
    logging_steps=200, 
    learning_rate=1e-5,  

    eval_steps=5000, 
    max_steps=-1, 
    num_train_epochs=1, 
    warmup_ratio=0.1, 
    lr_scheduler_type="cosine", 
    fp16=False,  
    bf16=True, 
    max_grad_norm=0.2, 
    gradient_checkpointing=True, 
    gradient_checkpointing_kwargs={'use_reentrant':False}, 

    dataset_text_field="prompt",
    max_seq_length=500, 
    packing=False,
    report_to=[],
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 8, 
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, 
    bias = "none", 
    use_gradient_checkpointing = "unsloth", 
    random_state = 3407,
    use_rslora = True, 
    loftq_config = None, 
)

model.gradient_checkpointing_enable()
tokenizer.pad_token = tokenizer.eos_token 

trainer = SFTTrainer(
        model=model,
        train_dataset=dataset,  
        tokenizer=tokenizer, 
        args=sft_config, 
)

trainer.model.print_trainable_parameters()

trainer.train()


PyTorch: setting up devices
Unsloth: Already have LoRA adapters! We shall skip this step.


Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Using auto half precision backend


trainable params: 5,636,096 || all params: 1,241,450,496 || trainable%: 0.4540


Currently training with a batch size of: 8
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs = 1
   \\   /|    Num examples = 2,000 | Num Epochs = 1
O^O/ \_/ \    Batch size per device = 8 | Gradient Accumulation steps = 8
\        /    Total batch size = 64 | Total steps = 31
 "-____-"     Number of trainable parameters = 5,636,096


Step,Training Loss


Saving model checkpoint to Lgb.Llama1B/checkpoint-31
loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--unsloth--llama-3.2-1b-instruct-unsloth-bnb-4bit/snapshots/8eb7999dc1d92775e7b5f75754a29c3fbdb32723/config.json
Model config LlamaConfig {
  "architectures": [
    "LlamaForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 128000,
  "eos_token_id": 128009,
  "head_dim": 64,
  "hidden_act": "silu",
  "hidden_size": 2048,
  "initializer_range": 0.02,
  "intermediate_size": 8192,
  "max_position_embeddings": 131072,
  "mlp_bias": false,
  "model_type": "llama",
  "num_attention_heads": 32,
  "num_hidden_layers": 16,
  "num_key_value_heads": 8,
  "pad_token_id": 128004,
  "pretraining_tp": 1,
  "quantization_config": {
    "_load_in_4bit": true,
    "_load_in_8bit": false,
    "bnb_4bit_compute_dtype": "bfloat16",
    "bnb_4bit_quant_storage": "uint8",
    "bnb_4bit_quant_type": "nf4",
    "bnb_4bit_use_double_qu

TrainOutput(global_step=31, training_loss=2.566320603893649, metrics={'train_runtime': 48.9296, 'train_samples_per_second': 40.875, 'train_steps_per_second': 0.634, 'total_flos': 3065217004634112.0, 'train_loss': 2.566320603893649, 'epoch': 0.992})